**ASSUMPTION OF REFERENCE ON UI**

In [ ]:
BOX_WIDTH_PIXELS = 800     # from UI design
BOX_HEIGHT_PIXELS = 800
REAL_BOX_WIDTH_CM = 20     # when held at correct fixed distance
REAL_BOX_HEIGHT_CM = 20


**Load Nutrint and Density Table**




In [ ]:
import pandas as pd
# --- 1. Load CSV from Google Drive ---
# If using Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# --- 1. Load CSV ---
csv_path = "pakistani_food_nutrients.csv"  # replace with your path
df = pd.read_csv(csv_path)

# --- 2. Function to retrieve density and nutrient info for a food ---
def get_food_info(food_name):
    """
    Returns a dictionary with density (g/cm3) and per-100g nutrients for the given food.
    """
    if food_name not in df['Food_Name'].values:
        print(f"Food '{food_name}' not found in database!")
        return None

    row = df[df['Food_Name'] == food_name].iloc[0]

    info = {
        'Density_g_per_cm3': row['Density_g_per_cm3'],
        'Calories_per_100g': row['Calories_per_100g'],
        'Protein_g_per_100g': row['Protein_g_per_100g'],
        'Carb_g_per_100g': row['Carb_g_per_100g'],
        'Fat_g_per_100g': row['Fat_g_per_100g'],
        'Fiber_g_per_100g': row.get('Fiber_g_per_100g', 0)
    }
    return info

**LOAD MODELS**

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.7 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import torch
import cv2
import numpy as np

def load_models(yolo_path):
    # Load YOLOv8 segmentation model
    yolo_model = YOLO(yolo_path)

    # Load MiDaS (Small or DPT_Large) from torch.hub
    # This automatically downloads weights if not present
    midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
    midas.eval()

    # Load transforms
    midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform

    return yolo_model, midas, midas_transforms


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


**YOLOv8 Segmentation → Mask Extraction**

In [ ]:
def segment_food(model, image_path):
    result = model(image_path, verbose=False)[0]

    # Store info for all detected objects
    foods = []

    # Loop over all detected masks
    for i in range(len(result.masks.data)):
        mask = result.masks.data[i].cpu().numpy()
        area_pixels = np.sum(mask == 1)

        # Bounding box
        x1, y1, x2, y2 = result.boxes.xyxy[i].cpu().numpy()
        width_px = x2 - x1
        height_px = y2 - y1

        # Class info
        class_id = int(result.boxes.cls[i].cpu().numpy())
        class_name = result.names[class_id]
        confidence = float(result.boxes.conf[i].cpu().numpy())

        foods.append({
            "mask": mask,
            "area_pixels": area_pixels,
            "width_px": width_px,
            "height_px": height_px,
            "class_id": class_id,
            "class_name": class_name,
            "confidence": confidence
        })

    return foods


**. Compute Scale Factor (due to fixed UI box distance)**

In [ ]:
def estimate_scale_factor():
    scale = REAL_BOX_WIDTH_CM / BOX_WIDTH_PIXELS
    return scale   # cm per pixel


**Convert YOLO Mask Area → Real Area (cm²)**

In [ ]:
def compute_area_cm2(area_pixels, scale):
    area_cm2 = area_pixels * (scale ** 2)
    return area_cm2


**MIDAS → Relative Depth → Real Height (cm)**

In [ ]:
def compute_height_cm_masked(image_path, midas, midas_transform, mask, scale):
    import cv2
    import torch
    import numpy as np

    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    input_batch = midas_transform(img_rgb).unsqueeze(0)

    with torch.no_grad():
        depth = midas(input_batch)[0].cpu().numpy()

    # Only consider pixels inside the mask
    masked_depth = depth * mask

    # Take maximum height
    relative_height_px = np.max(masked_depth)

    height_cm = relative_height_px * scale
    return height_cm


**Compute Volume (cm³)**

In [ ]:
def compute_volume_cm3(area_cm2, height_cm):
    return area_cm2 * height_cm


**Convert Volume → Grams**

In [ ]:
def estimate_grams(volume_cm3, food_name, density_table):
    density = density_table.get(food_name, 1.0)  # default
    return volume_cm3 * density



**Full Pipeline Function**

In [ ]:
def food_portion_pipeline_multi(image_path):
    # Load models
   yolo_model, midas, midas_transforms = load_models(
    "/content/drive/MyDrive/your_yolo/best.pt"
)

    # Step 1 — Multi-food segmentation
    detected_foods = segment_food(yolo_model, image_path)

    # Step 2 — Scale factor (same for all foods)
    scale = estimate_scale_factor()

    results = []

    # Step 3 — Loop over each detected food
    for food in detected_foods:
        mask = food["mask"]
        area_pixels = food["area_pixels"]
        class_name = food["class_name"]
        confidence = food["confidence"]

        # Real area
        area_cm2 = compute_area_cm2(area_pixels, scale)

        # Height — compute using MIDAS inside mask
        height_cm = compute_height_cm_masked(image_path, midas, midas_transforms, mask, scale)

        # Volume
        volume_cm3 = compute_volume_cm3(area_cm2, height_cm)
        info = get_food_info(food_name)

        # Grams using density
        grams = estimate_grams(volume_cm3, class_name, info['Density_g_per_cm3'])

        results.append({
            "food_class": class_name,
            "confidence": confidence,
            "area_pixels": area_pixels,
            "area_cm2": area_cm2,
            "height_cm": height_cm,
            "volume_cm3": volume_cm3,
            "grams": grams
        })

    return results


**Nutrient Profiling**

In [ ]:
def compute_nutrients(food_name, portion_grams):
    """
    Computes nutrients based on estimated portion in grams.
    """
    info = get_food_info(food_name)
    if info is None:
        return None

    factor = portion_grams / 100
    nutrients = {
        'Food': food_name,
        'Portion_g': portion_grams,
        'Calories': round(info['Calories_per_100g'] * factor, 2),
        'Protein_g': round(info['Protein_g_per_100g'] * factor, 2),
        'Carbs_g': round(info['Carb_g_per_100g'] * factor, 2),
        'Fat_g': round(info['Fat_g_per_100g'] * factor, 2),
        'Fiber_g': round(info['Fiber_g_per_100g'] * factor, 2)
    }
    return nutrients